# **Activity 2: Building the Data Processing Pipeline - Group 3**

**Group members:**
- Abdullah Ali Saleem
- Danil Seksenov
- Nikita Soo
- Fatih Özdil

##**Setup: Environment Preparation**

In [ ]:
# Install the necessary libraries
!pip install datasets pandas -q

**Group Discussion 1**

---

**Question:** In the code above, we used an exclamation mark (`!`) before `pip install`. In Colab, you might also occasionally see commands starting with a percent sign (`%` or `%%`). What is the difference between `!` and `%`, and why do we use them?

### ! Exclamation mark is for shell no the python enviroments
### % %% these marks Jupyter/Colab notebooks to control how the notebook behaves

##**Part 1: Inspecting and Deduplicating Data**

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the dataset
dataset = load_dataset('fka/awesome-chatgpt-prompts', split='train')

# Convert to Pandas DataFrame
df = dataset.to_pandas()

print(f"Total number of rows before removing duplicates: {len(df)}")

# Remove duplicate rows based on the 'prompt' column
df_cleaned = df.drop_duplicates(subset=['prompt'])

print(f"Total number of rows after removing duplicates: {len(df_cleaned)}")

# Display the first few rows of the cleaned DataFrame
display(df_cleaned.head())

**Group Discussion 2**

---

**Question:** The code above successfully removed *exact* duplicate prompts. In the context of LLM training, why might dropping exact duplicates not be enough? What other types of duplicates exist that we should worry about?


###"Because you can write same  promte in  two different ways"

##**Part 2: Cleaning and Filtering Data**

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the dataset
dataset = load_dataset('rotten_tomatoes', split='train')

# Convert to Pandas DataFrame
df_rotten = dataset.to_pandas()

print(f"Initial number of rows: {len(df_rotten)}")

# Filter out reviews shorter than 30 characters
df_rotten_filtered = df_rotten[df_rotten['text'].str.len() >= 30]

print(f"Number of rows after filtering short reviews: {len(df_rotten_filtered)}")

# Map numerical labels to 'negative' and 'positive'
label_mapping = {0: 'negative', 1: 'positive'}
df_rotten_filtered['label'] = df_rotten_filtered['label'].map(label_mapping)

# Display the first few rows of the processed DataFrame
display(df_rotten_filtered.head())

**Group Discussion 3**

---

**Question:** Why did we decide to drop reviews that were shorter than 30 characters? Think about the "Garbage In, Garbage Out" principle from the reading material.

###Because Reviews shorter than 30 characters lack sufficient context or reasoning

##**Part 3: Formatting Data (Building Chat Templates)**

In [ ]:
from datasets import load_dataset
import pandas as pd

# 1. Load the dataset
dataset_dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
df_dolly = dataset_dolly.to_pandas()

# 2. Define the formatting function
def format_templates(row):
    instruction = row['instruction']
    response = row['response']

    # f-strings allow us to inject variables inside curly brackets {}. \n creates a line break.

    # Gemma Format
    gemma = f"<start_of_turn>user\n{instruction}<end_of_turn>\n<start_of_turn>model\n{response}<end_of_turn>"

    # Llama 3 Format
    llama3 = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>"

    # Return as a Pandas Series so it creates two separate columns
    return pd.Series([gemma, llama3])

# 3. Apply the function to the DataFrame
df_dolly[['gemma_formatted', 'llama3_formatted']] = df_dolly.apply(format_templates, axis=1)

# 4. Print to compare the differences!
print("--- GEMMA FORMAT ---")
print(df_dolly['gemma_formatted'].iloc[0])
print("\n--- LLAMA 3 FORMAT ---")
print(df_dolly['llama3_formatted'].iloc[0])

**Group Discussion 4**

---

**Question:** Look at the printout comparing the Gemma format to the Llama 3 format. They look completely alien to each other! What would happen if you successfully fine-tuned a Llama 3 model on the `llama3_formatted` column, but when you deployed it to your app, your app's code sent user messages using the `gemma_formatted` template?

###Because if you fine-tune a model on one chat template (Llama 3) but then use a different template (Gemma) in your application, the model will not understand the input format, leading to poor or nonsensical responses.